In [3]:
import pandas as pd
import numpy as np

train_fe = pd.read_parquet("../data/processed/train_fe.parquet")
val_fe = pd.read_parquet("../data/processed/val_fe.parquet")

print(train_fe.shape, val_fe.shape)

(5661993, 21) (85372, 21)


In [4]:
target = "sales"

exclude_cols = [
    "sales", "date", "item_id", "dept_id",
    "cat_id", "store_id", "state_id"
]

features = [col for col in train_fe.columns if col not in exclude_cols]

X_train = train_fe[features]
y_train = train_fe[target]

X_val = val_fe[features]
y_val = val_fe[target]

print("Number of features:", len(features))

Number of features: 14


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    # SMAPE: stable when y_true has zeros
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    smape = np.mean(np.where(denom == 0, 0, np.abs(y_true - y_pred) / denom)) * 100

    return rmse, mae, smape

In [6]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

lr_preds = lr.predict(X_val)

lr_rmse, lr_mae, lr_mape = evaluate(y_val, lr_preds)

print("Linear Regression")
print("RMSE:", lr_rmse)
print("MAE:", lr_mae)
print("MAPE:", lr_mape)

Linear Regression
RMSE: 2.121895119963127
MAE: 1.084166882246089
MAPE: 136.6500420366244


In [6]:
pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.6 MB/s eta 0:00:00m eta 0:00:010:01
Note: you may need to restart the kernel to use updated packages.


In [7]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(X_train, y_train)
lgb_preds = lgb_model.predict(X_val)

lgb_rmse, lgb_mae, lgb_smape = evaluate(y_val, lgb_preds)

print("Default LightGBM")
print("RMSE:", lgb_rmse)
print("MAE:", lgb_mae)
print("SMAPE:", lgb_smape)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.089484 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1373
[LightGBM] [Info] Number of data points in the train set: 5661993, number of used features: 14
[LightGBM] [Info] Start training from score 1.320919
Default LightGBM
RMSE: 2.03778138333104
MAE: 1.0457566889423946
SMAPE: 134.7496093038735


In [ ]:
lgb_preds = lgb_model.predict(X_val)
lgb_rmse, lgb_mae, lgb_smape = evaluate(y_val, lgb_preds)

print("Default LightGBM")
print("RMSE:", lgb_rmse)
print("MAE:", lgb_mae)
print("SMAPE:", lgb_smape)